# Walidacja wektorów — dlaczego retrieval nie trafia?

Po uruchomieniu lekcji 14 (Qdrant Hybrid) widzimy że niektóre produkty **nigdy nie trafiają do kandydatów**
mimo że powinny być oczywistą odpowiedzią:

| Pytanie | Produkt | Coverage |
|---------|---------|----------|
| Q01 — żel do toalety | Zurada Sanit Żel (12) | 0/8 |
| Q02 — nagar z grilla | Zurada Nagar Moc (5), Zurada Piekarnik Plus (34) | 2/8 |
| Q03 — aluminium warsztat | Zurada Aluminium Plus (45), Zurada Metal Wash (112), Zurada Warsztat Pro (152) | 0–1/8 |

## Co sprawdzamy w tej lekcji

1. **embed_text** — co dokładnie jest osadzone jako wektor dla brakujących produktów
2. **Cosine similarity** — jak blisko pytania są brakujące vs znajdowane produkty
3. **Rank w dense i sparse** — gdzie brakujący produkt ląduje w osobnych listach
4. **BM25 token overlap** — czy sparse w ogóle widzi wspólne słowa
5. **Hipoteza: embed_text jest za słaby** — co możemy poprawić

In [5]:
import os
import re
import json
import numpy as np
from pathlib import Path

from openai import OpenAI
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import SparseVector

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "Brak klucza")
EMBED_MODEL  = "openai/text-embedding-3-small"
SPARSE_MODEL = "Qdrant/bm25"
COLLECTION   = "zurada_products_hybrid"

BASE_DIR   = Path(".")
VAULT_DIR  = BASE_DIR / "../11_vault_rag_improvements/zurada_vault"
QDRANT_PATH = BASE_DIR / "../14_qdrant_hybrid/qdrant_hybrid_storage"

embed_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
bm25         = SparseTextEmbedding(model_name=SPARSE_MODEL)
qdrant       = QdrantClient(path=str(QDRANT_PATH))

print(f"Collection: {COLLECTION}  ({qdrant.count(collection_name=COLLECTION).count} points)")
print(f"API key: {'OK' if OPENROUTER_API_KEY != 'Brak klucza' else 'BRAK'}")

Collection: zurada_products_hybrid  (262 points)
API key: OK


In [15]:
def embed_one(text: str) -> list[float]:
    return embed_client.embeddings.create(model=EMBED_MODEL, input=[text]).data[0].embedding


def sparse_one(text: str) -> dict:
    e = list(bm25.embed([text]))[0]
    return {"indices": e.indices.tolist(), "values": e.values.tolist()}


def cosine(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def sparse_dot(a: dict, b: dict) -> float:
    """Dot product of two sparse vectors."""
    a_map = dict(zip(a["indices"], a["values"]))
    return sum(a_map.get(i, 0.0) * v for i, v in zip(b["indices"], b["values"]))


def get_point(product_id: int) -> dict:
    pts = qdrant.retrieve(
        collection_name=COLLECTION,
        ids=[product_id],
        with_vectors=True,
        with_payload=True,
    )
    return pts[0] if pts else None


def dense_rank(q_vec: list[float], product_id: int, limit: int = 262) -> int:
    """Find the rank of product_id in dense search (1 = best)."""
    resp = qdrant.query_points(
        collection_name=COLLECTION,
        query=q_vec,
        using="dense",
        limit=limit,
        with_payload=False,
    )
    print(resp)
    resp = qdrant.query_points(
        collection_name=COLLECTION,
        query=q_vec,
        using="dense",
        limit=limit,
        with_payload=False,
    )
    print(resp)
    for rank, pt in enumerate(resp.points, 1):
        if pt.id == product_id:
            return rank
    return limit + 1  # not found in top-limit


def sparse_rank(q_sparse: dict, product_id: int, limit: int = 262) -> int:
    """Find the rank of product_id in sparse (BM25) search."""
    resp = qdrant.query_points(
        collection_name=COLLECTION,
        query=SparseVector(indices=q_sparse["indices"], values=q_sparse["values"]),
        using="sparse",
        limit=limit,
        with_payload=False,
    )
    for rank, pt in enumerate(resp.points, 1):
        if pt.id == product_id:
            return rank
    return limit + 1


print("Helper functions defined")

Helper functions defined


---
## Analiza Q01 — żel do toalety

Produkt 12 (Zurada Sanit Żel) nigdy nie trafia do top-20.  
Produkty 82 i 139 trafiają zawsze.  
Co je odróżnia?

In [17]:
Q01 = "Potrzebuję zwykłego, gęstego żelu do codziennego mycia toalety w domu, żeby usunąć trochę kamienia i odświeżyć muszlę."

q_dense  = embed_one(Q01)
q_sparse = sparse_one(Q01)

print(f"Pytanie: {Q01}")
print(f"BM25 tokens w pytaniu: {len(q_sparse['indices'])}")


# Compare products 12, 82, 139
TARGET_IDS = [12, 82, 139]

for pid in TARGET_IDS:
    pt    = get_point(pid)
    name  = pt.payload["name"]
    cos   = cosine(q_dense, pt.vector["dense"])
    sp    = {"indices": pt.vector["sparse"].indices, "values": pt.vector["sparse"].values}
    sp_score = sparse_dot(q_sparse, sp)
    d_rank = dense_rank(q_dense, pid, limit=30)
    s_rank = sparse_rank(q_sparse, pid, limit=30)

    print(f"[{pid:3}] {name}")
    print(f"      dense cosine: {cos:.4f}   dense rank (top-30): {d_rank if d_rank <= 30 else '>30'}")
    print(f"      sparse score: {sp_score:.4f}  sparse rank (top-30): {s_rank if s_rank <= 30 else '>30'}")
    print()

Pytanie: Potrzebuję zwykłego, gęstego żelu do codziennego mycia toalety w domu, żeby usunąć trochę kamienia i odświeżyć muszlę.
BM25 tokens w pytaniu: 15
points=[ScoredPoint(id=139, version=0, score=0.6460035608667789, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=79, version=0, score=0.6329050123035869, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=12, version=0, score=0.6267875855181221, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=82, version=0, score=0.6245222003869526, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=80, version=0, score=0.6185577878554368, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=54, version=0, score=0.6006159321111804, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=138, version=0, score=0.5974168019822301, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoin

In [5]:
# Show embed_text for the three products
def get_embed_text(product_id: int) -> tuple[str, str]:
    pt = get_point(product_id)
    name = pt.payload["name"]
    path = VAULT_DIR / "Produkty" / f"{name}.md"
    text = path.read_text(encoding="utf-8")

    # Reproduce embed_text logic from lesson 14
    title = re.search(r'^title:\s*"?(.+?)"?\s*$', text, re.MULTILINE)
    title = title.group(1) if title else name
    parts = text.split("---", 2)
    body  = parts[2] if len(parts) >= 3 else text
    body  = re.sub(r'^\*\*[^*]+\*\*.*$', '', body, flags=re.MULTILINE)
    body  = re.sub(r'\[\[.*?\]\]', '', body)
    body  = re.sub(r'#+\s+', '', body)
    body  = re.sub(r'[⛔>]', '', body)
    body  = ' '.join(body.split())
    return name, f"{title}. {body}"


print("=" * 70)
for pid in TARGET_IDS:
    name, embed_text = get_embed_text(pid)
    print(f"[{pid}] {name}")
    print(f"{embed_text[:300]}")
    print()

[12] Zurada Sanit Żel
Żel do WC i sanitariatów. Zurada Sanit Żel Zagęszczony preparat do codziennego czyszczenia i odkamieniania toalet. --- - - --- Opis Zagęszczony żel do ręcznego czyszczenia toalet, pisuarów, wanien i brodzików. Skutecznie usuwa osady kamienne i zabrudzenia, a także wspiera bieżące utrzymanie czystośc

[82] Zurada Toaleta Żel
Żel do mycia toalet Eco. Zurada Toaleta Żel Ekologiczny, skoncentrowany żel do czyszczenia i odkamieniania toalet oraz sanitariatów. --- --- Opis Ekologiczny żel do mycia toalet i innych powierzchni sanitarnych, który skutecznie czyści oraz usuwa kamień. Sprawdza się przy muszlach klozetowych, pisua

[139] Zurada WC Żel Dom
Żel do WC. Zurada WC Żel Dom Gęsty żel do czyszczenia toalet i ceramiki sanitarnej, usuwa kamień, rdzę oraz osady. --- - - - --- Opis Zurada Sanit Żel to skuteczny preparat do czyszczenia muszli WC oraz innych urządzeń sanitarnych. Gęsta formuła dobrze przylega do pionowych powierzchni, ułatwiając u



---
## Dlaczego produkt 12 jest na dense rank 3, ale nie trafia do wyników?

Produkt 12 jest w dense top-20 (rank 3) — powinien wejść do puli RRF.  
Sprawdzamy co dokładnie dzieje się w środku: które produkty wchodzą z obu list  
i gdzie ląduje produkt 12 po fuzji.

In [8]:
from qdrant_client.models import Filter, FieldCondition, MatchAny

# Reproduce the exact payload filter used by Q01 in the pipeline
# (intent: surfaces=['toalety'], categories=['zele-do-wc','higiena-toalet','odkamienianie','dla-domu'])
q01_filter = Filter(should=[
    FieldCondition(key="kategorie", match=MatchAny(any=["zele-do-wc", "higiena-toalet", "odkamienianie", "dla-domu"])),
    FieldCondition(key="surfaces",  match=MatchAny(any=["toalety"])),
])

TOP_K = 20

# Dense Prefetch with filter
dense_resp = qdrant.query_points(
    COLLECTION, query=q_dense, using="dense",
    query_filter=q01_filter, limit=TOP_K, with_payload=True,
)
# Sparse Prefetch with filter
sparse_resp = qdrant.query_points(
    COLLECTION,
    query=SparseVector(indices=q_sparse["indices"], values=q_sparse["values"]),
    using="sparse", query_filter=q01_filter, limit=TOP_K, with_payload=True,
)

dense_ids  = [p.id for p in dense_resp.points]
sparse_ids = [p.id for p in sparse_resp.points]

print(f"Dense top-{TOP_K}  (z filtrem): {dense_ids}")
print(f"Sparse top-{TOP_K} (z filtrem): {sparse_ids}")
print()
print(f"Produkt 12 w dense:  {12 in dense_ids}  (rank {dense_ids.index(12)+1 if 12 in dense_ids else 'N/A'})")
print(f"Produkt 12 w sparse: {12 in sparse_ids}")
print()

# Manually compute RRF on the union of both lists
K_RRF = 60
rrf_scores = {}
rrf_payloads = {}
for rank, p in enumerate(dense_resp.points, 1):
    rrf_scores[p.id]   = rrf_scores.get(p.id, 0.0) + 1.0 / (K_RRF + rank)
    rrf_payloads[p.id] = p.payload
for rank, p in enumerate(sparse_resp.points, 1):
    rrf_scores[p.id]   = rrf_scores.get(p.id, 0.0) + 1.0 / (K_RRF + rank)
    rrf_payloads[p.id] = p.payload

ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

TARGETS = {12, 82, 139}
print(f"{'RRF rank':>8}  {'ID':>4}  {'Nazwa':<32}  {'RRF score':>10}  {'dense_r':>7}  {'sparse_r':>8}")
print("─" * 80)
for rrf_rank, (pid, score) in enumerate(ranked, 1):
    d_r = dense_ids.index(pid)+1  if pid in dense_ids  else "—"
    s_r = sparse_ids.index(pid)+1 if pid in sparse_ids else "—"
    tag = "  ← EXPECTED" if pid in TARGETS else ""
    name = rrf_payloads[pid]["name"]
    print(f"{rrf_rank:>8}  {pid:>4}  {name:<32}  {score:>10.6f}  {str(d_r):>7}  {str(s_r):>8}{tag}")

Dense top-20  (z filtrem): [139, 79, 12, 82, 80, 129, 133, 2, 29, 68, 229, 4, 41, 11, 33, 46, 39, 92, 10, 93]
Sparse top-20 (z filtrem): [216, 190, 189, 129, 46, 211, 137, 188, 229, 130, 219, 136, 135, 215, 220, 226, 11, 12, 33, 212]

Produkt 12 w dense:  True  (rank 3)
Produkt 12 w sparse: True

RRF rank    ID  Nazwa                              RRF score  dense_r  sparse_r
────────────────────────────────────────────────────────────────────────────────
       1   129  Zurada Szkło Klar                   0.030777        6         4
       2    12  Zurada Sanit Żel                    0.028694        3        18  ← EXPECTED
       3   229  Zurada Srebrny Blask                0.028577       11         9
       4    46  Zurada Posadzka Lux                 0.028543       16         5
       5    11  Zurada Czysta Łazienka              0.026501       14        17
       6    33  Zurada Kwas Pro                     0.025992       15        19
       7   139  Zurada WC Żel Dom                

---
## Gdzie ginie produkt 12 — safety filter czy recommend?

Retrieval działa (rank 2 po RRF). Ale `retrieved=20→15` w logach oznacza że safety filter  
usuwa 5 produktów. Sprawdzamy czy produkt 12 jest wśród nich i dlaczego.

In [9]:
VAULT_DIR_STR = BASE_DIR / "../11_vault_rag_improvements/zurada_vault"

# Check product 12's odradzane — did we add marmur in lesson 11?
pt12  = get_point(12)
name12 = pt12.payload["name"]
odradzane12 = pt12.payload.get("odradzane", [])

print(f"Produkt 12: {name12}")
print(f"Odradzane w payload Qdrant: {odradzane12}")
print()

# Read the actual vault file odradzane
vault_file = VAULT_DIR_STR / "Produkty" / f"{name12}.md"
text = vault_file.read_text(encoding="utf-8")
odradzane_line = [l for l in text.splitlines() if "Odradzane" in l]
print(f"Odradzane w pliku vault:")
for l in odradzane_line:
    print(f"  {l}")
print()

# Hypothesis: we added marmur to product 12's odradzane in lesson 11
# to fix Q04 (marble bathroom) — which may now cause safety filter to exclude it for Q01
print("Hipoteza: w lekcji 11 dodano 'marmur' do odradzane produktu 12")
print("żeby acid descaler nie był polecany na marmur.")  
print("Safety LLM może teraz błędnie wykluczać produkt 12 dla pytań o łazienkę.")
print()
print("Sprawdź wynik safety filter poniżej — uruchom safety LLM dla Q01")
print("z kontekstem zawierającym produkt 12 i sprawdź czy go wyklucza.")

Produkt 12: Zurada Sanit Żel
Odradzane w payload Qdrant: ['powierzchnie wrażliwe na kwasy', 'aluminium', 'marmur', 'kamień naturalny']

Odradzane w pliku vault:
  **Odradzane powierzchnie:** ⛔ [[Odradzane/powierzchnie wrażliwe na kwasy|powierzchnie wrażliwe na kwasy]] · [[Odradzane/aluminium|aluminium]] · [[Odradzane/marmur|marmur]] · [[Odradzane/kamień naturalny|kamień naturalny]]

Hipoteza: w lekcji 11 dodano 'marmur' do odradzane produktu 12
żeby acid descaler nie był polecany na marmur.
Safety LLM może teraz błędnie wykluczać produkt 12 dla pytań o łazienkę.

Sprawdź wynik safety filter poniżej — uruchom safety LLM dla Q01
z kontekstem zawierającym produkt 12 i sprawdź czy go wyklucza.


In [17]:
import instructor
from pydantic import BaseModel, Field
from typing import List

_llm_raw = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
llm = instructor.from_openai(_llm_raw, mode=instructor.Mode.JSON)

# EXACT same prompt as qdrant_hybrid_rag.ipynb cell-14
SAFETY_SYSTEM_HYBRID = """Jesteś ekspertem chemicznym oceniającym bezpieczeństwo środków czyszczących.

Otrzymasz strony produktów z bazy Zurada oraz pytanie klienta.

Twoje jedyne zadanie: zidentyfikuj produkty NIEBEZPIECZNE dla opisanego kontekstu — nie filtruj
pod kątem trafności czy dopasowania (to robi osobny krok).

Wyklucz produkt (zwróć jego product_id) TYLKO jeśli zachodzi jeden z poniższych warunków:
1. Jego sekcja "Odradzane powierzchnie" zawiera powierzchnię WPROST wymienioną w pytaniu klienta.
2. Skład/opis wskazuje na KONKRETNĄ, BEZPOŚREDNIĄ niezgodność chemiczną z kontekstem
   (przykłady: silny kwas na marmur lub kamień naturalny, podchloryn na aluminium lub tkaniny).
3. Ostrzeżenia BHP tego produktu wprost wykluczają opisany sposób użycia.

WAŻNE — produkty higieny osobistej i mydła:
Mydła w płynie, żele do mycia rąk, szampony i środki do higieny osobistej (kategorie vault:
'mydla-w-plynie', 'higiena-rak', 'higiena-osobista', 'mycie-rak') są z definicji bezpieczne
dla skóry i typowych powierzchni łazienkowych. Wyklucz je wyłącznie gdy kryterium 1 jest
spełnione — tzn. gdy ich "Odradzane powierzchnie" zawiera konkretną powierzchnię z pytania.
Nie wykluczaj ich na podstawie kryterium 2 ani 3.

ZASADY OGÓLNE:
- Ten filtr ocenia wyłącznie BEZPIECZEŃSTWO, nie trafność produktu.
- Nie wykluczaj produktu tylko dlatego, że jest silny, profesjonalny lub nie najlepiej pasuje.
- Gdy wątpliwość — zachowaj produkt (nie wykluczaj)."""


class SafetyResult(BaseModel):
    excluded_ids: List[int] = Field(default_factory=list)
    reasoning: List[str]    = Field(default_factory=list)


# Run safety filter with ONLY the 3 expected products
debug_pids = [12, 82, 139]
debug_pages = []
for pid in debug_pids:
    pt   = get_point(pid)
    name = pt.payload["name"]
    path = VAULT_DIR_STR / "Produkty" / f"{name}.md"
    debug_pages.append(path.read_text(encoding="utf-8"))
debug_context = "\n\n---\n\n".join(debug_pages)

result, _ = llm.chat.completions.create_with_completion(
    model="google/gemini-3.1-flash-lite",
    messages=[
        {"role": "system", "content": SAFETY_SYSTEM_HYBRID},
        {"role": "user",   "content": f"Strony produktów:\n\n{debug_context}\n\n---\n\nPytanie klienta: {Q01}"},
    ],
    response_model=SafetyResult,
    max_tokens=500,
    temperature=0,
)

print(f"Pytanie: {Q01}")
print(f"Produkty w teście: {debug_pids}")
print(f"Excluded IDs: {result.excluded_ids}")
print()
print("Reasoning:")
for r in result.reasoning:
    print(f"  - {r}")
print()
print("─" * 60)
print("Jeśli excluded_ids=[] → problem jest w RECOMMEND, nie w safety")
print("Jeśli excluded_ids=[12] → problem jest w SAFETY_SYSTEM kryterium 2")

Pytanie: Potrzebuję zwykłego, gęstego żelu do codziennego mycia toalety w domu, żeby usunąć trochę kamienia i odświeżyć muszlę.
Produkty w teście: [12, 82, 139]
Excluded IDs: []

Reasoning:
  - Wszystkie produkty (12, 82, 139) są przeznaczone do czyszczenia toalet i usuwania kamienia, co jest zgodne z zapytaniem klienta. Żaden z produktów nie zawiera w sekcji 'Odradzane powierzchnie' materiałów, które byłyby wprost wymienione w pytaniu (klient pyta o muszlę toaletową, która jest standardową powierzchnią dla tych produktów). Nie stwierdzono bezpośredniej niezgodności chemicznej ani naruszenia zasad BHP w kontekście opisanego zastosowania domowego.

────────────────────────────────────────────────────────────
Jeśli excluded_ids=[] → problem jest w RECOMMEND, nie w safety
Jeśli excluded_ids=[12] → problem jest w SAFETY_SYSTEM kryterium 2


---
## Analiza Q03 — aluminium warsztat

Produkty 45, 112, 152 prawie nigdy nie trafiają do top-20.  
Sprawdzamy ranking i czy BM25 widzi słowo 'aluminium'.

In [6]:
with open(BASE_DIR / "../5_navie_rag_extended_data/extended_golden_questions.json") as f:
    questions = json.load(f)

Q03 = next(q["question"] for q in questions if q["id"] == 3)
print(f"Q03: {Q03}")
print()

q3_dense  = embed_one(Q03)
q3_sparse = sparse_one(Q03)

print(f"BM25 tokens w pytaniu: {len(q3_sparse['indices'])}")
print()

# Expected products + one always-returned bad extra (44)
for pid in [45, 112, 152, 44]:
    pt    = get_point(pid)
    name  = pt.payload["name"]
    cos   = cosine(q3_dense, pt.vector["dense"])
    sp    = {"indices": pt.vector["sparse"].indices, "values": pt.vector["sparse"].values}
    sp_score = sparse_dot(q3_sparse, sp)
    d_rank = dense_rank(q3_dense, pid, limit=30)
    s_rank = sparse_rank(q3_sparse, pid, limit=30)

    tag = "  ← OCZEKIWANY" if pid in [45, 112, 152] else "  ← ZŁY EXTRA (no-list)"
    print(f"[{pid:3}] {name}{tag}")
    print(f"      dense cosine: {cos:.4f}   dense rank (top-30): {d_rank if d_rank <= 30 else '>30'}")
    print(f"      sparse score: {sp_score:.4f}  sparse rank (top-30): {s_rank if s_rank <= 30 else '>30'}")
    print()

Q03: Potrzebuję mocnego środka do odtłuszczenia narzędzi warsztatowych oraz aluminiowych części maszyn. Zależy mi, żeby środek był w 100% bezpieczny – część z tych elementów jest lakierowana.

BM25 tokens w pytaniu: 24

[ 45] Zurada Odtłuszczacz Moc  ← OCZEKIWANY
      dense cosine: 0.5434   dense rank (top-30): >30
      sparse score: 13.6140  sparse rank (top-30): 27

[112] Zurada Smar Stop  ← OCZEKIWANY
      dense cosine: 0.5529   dense rank (top-30): 23
      sparse score: 20.4623  sparse rank (top-30): 2

[152] Zurada Odtłuszczacz Uni  ← OCZEKIWANY
      dense cosine: 0.5169   dense rank (top-30): >30
      sparse score: 11.8408  sparse rank (top-30): >30

[ 44] Zurada Silnik Moc  ← ZŁY EXTRA (no-list)
      dense cosine: 0.6173   dense rank (top-30): 1
      sparse score: 13.7281  sparse rank (top-30): 25



In [ ]:
# Show top-10 dense results for Q03 to understand what's winning
print("TOP-10 dense wyniki dla Q03:")
resp = qdrant.query_points(
    collection_name=COLLECTION,
    query=q3_dense,
    using="dense",
    limit=10,
    with_payload=True,
)
expected_q03 = {45, 112, 152}
for rank, pt in enumerate(resp.points, 1):
    tag = "  ← OCZEKIWANY" if pt.id in expected_q03 else ""
    print(f"  rank {rank:2d}: [{pt.id:3d}] {pt.payload['name']:35s}  score={pt.score:.4f}{tag}")

print()
print("TOP-10 sparse (BM25) wyniki dla Q03:")
resp_s = qdrant.query_points(
    collection_name=COLLECTION,
    query=SparseVector(indices=q3_sparse["indices"], values=q3_sparse["values"]),
    using="sparse",
    limit=10,
    with_payload=True,
)
for rank, pt in enumerate(resp_s.points, 1):
    tag = "  ← OCZEKIWANY" if pt.id in expected_q03 else ""
    print(f"  rank {rank:2d}: [{pt.id:3d}] {pt.payload['name']:35s}  score={pt.score:.4f}{tag}")

---
## Hipoteza: HyDE (Hypothetical Document Embedding)

Zamiast osadzać pytanie klienta (krótkie, konwersacyjne), LLM generuje
**hipotetyczny opis produktu** pasujący do pytania i to jego embedding
trafia do Qdrant.

Hipoteza: tekst wygenerowany przez LLM brzmi jak opis produktu → jest
semantycznie bliżej opisów w kolekcji niż surowe pytanie.

```
Standardowe:  embed("Czym usunąć nagar z grilla?")  → daleko od opisów produktów
HyDE:         LLM → "Środek alkaliczny do usuwania nagaru..."
              embed("Środek alkaliczny...") → blisko opisów produktów
```

In [ ]:
from openai import OpenAI
import instructor
from pydantic import BaseModel

_llm_raw = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
llm = instructor.from_openai(_llm_raw, mode=instructor.Mode.JSON)


class HyDE(BaseModel):
    hypothetical_product: str


HYDE_SYSTEM = """Jesteś ekspertem od środków czyszczących Zurada.
Na podstawie pytania klienta napisz krótki (2-3 zdania) OPIS PRODUKTU który by odpowiadał na to pytanie.
Pisz jak opis produktu z katalogu — skup się na składzie, przeznaczeniu i właściwościach.
NIE odpowiadaj na pytanie — tylko opisz produkt który by pasował."""


def hyde(question: str) -> str:
    result = llm.chat.completions.create(
        model="google/gemini-3.1-flash-lite",
        messages=[
            {"role": "system", "content": HYDE_SYSTEM},
            {"role": "user",   "content": question},
        ],
        response_model=HyDE,
        max_tokens=200,
        temperature=0,
    )
    return result.hypothetical_product


# Test on Q01 and Q03
for q_text, label, expected_ids in [
    (Q01, "Q01", [12, 82, 139]),
    (Q03, "Q03", [45, 112, 152]),
]:
    hypo = hyde(q_text)
    print(f"=== {label} ===")
    print(f"Pytanie:    {q_text}")
    print(f"HyDE opis:  {hypo}")
    print()

In [ ]:
# Compare cosine: raw question vs HyDE for missing products
print("Porównanie cosine: surowe pytanie vs HyDE\n")

for q_text, label, pids in [
    (Q01, "Q01", [12, 82, 139]),
    (Q03, "Q03", [45, 112, 152]),
]:
    hypo      = hyde(q_text)
    q_vec     = embed_one(q_text)
    hypo_vec  = embed_one(hypo)

    print(f"{'─'*60}")
    print(f"{label}  HyDE: {hypo[:80]}...")
    print(f"{'─'*60}")
    print(f"  {'Produkt':<35} {'cosine(raw)':>12} {'cosine(HyDE)':>13} {'delta':>7}")

    for pid in pids:
        pt   = get_point(pid)
        name = pt.payload["name"]
        pv   = pt.vector["dense"]
        c_raw  = cosine(q_vec, pv)
        c_hyde = cosine(hypo_vec, pv)
        delta  = c_hyde - c_raw
        arrow  = "↑" if delta > 0.01 else ("↓" if delta < -0.01 else "→")
        print(f"  [{pid:3}] {name:<30} {c_raw:12.4f} {c_hyde:13.4f} {arrow}{delta:+.4f}")
    print()

---
## Wnioski

Na podstawie analizy powyżej możemy określić:

1. **Czy problem jest w embed_text produktu** (opis zbyt ubogi, brak kluczowych słów)
2. **Czy problem jest w embeddingu pytania** (konwersacyjne pytanie daleko od opisów produktów)
3. **Czy BM25 widzi wspólne tokeny** (Polish stemming może powodować brak matchów)
4. **Czy HyDE poprawia cosine** dla brakujących produktów → czy warto wprowadzić HyDE w lekcji 16

In [1]:
qdrant.close()

NameError: name 'qdrant' is not defined